In [ ]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer # Import the Breast Cancer dataset
from sklearn.cluster import KMeans

# Step 1: Load and Preprocess the Dataset
# Load the Breast Cancer dataset
cancer = load_breast_cancer()
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)

# Selecting relevant features: Mean radius and Mean smoothness
df = df[['mean radius', 'mean smoothness']]

# Scale the features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df)

# Step 2: Generate Pseudo-Labels for Supervised Evaluation
# Use K-Means clustering to create pseudo-labels
kmeans = KMeans(n_clusters=2, random_state=42)
kmeans_labels = kmeans.fit_predict(scaled_features)

# Step 3: Split Dataset into Labeled and Unlabeled Subsets
X_labeled, X_unlabeled, y_labeled, y_unlabeled = train_test_split(
    scaled_features, kmeans_labels, test_size=0.7, random_state=42, stratify=kmeans_labels
)

# Step 4: Train a Gaussian Mixture Model (GMM)
# Train the GMM on the labeled subset
gmm = GaussianMixture(n_components=2, random_state=42)
gmm.fit(X_labeled)

# Evaluate on labeled data
labeled_predictions = gmm.predict(X_labeled)
labeled_acc = accuracy_score(y_labeled, labeled_predictions)
print(f"Accuracy on labeled data: {labeled_acc * 100:.2f}%")

# Step 5: Predict Clusters for Unlabeled Data
# Use the trained GMM to predict clusters for the unlabeled subset
unlabeled_predictions = gmm.predict(X_unlabeled)

# Step 6: Map Clusters to Pseudo-Labels
# Map the predicted clusters to their majority pseudo-label class
cluster_to_class = {}
for cluster in np.unique(unlabeled_predictions):
    mask = unlabeled_predictions == cluster
    majority_class = np.bincount(y_unlabeled[mask]).argmax()
    cluster_to_class[cluster] = majority_class

final_unlabeled_predictions = np.array(
    [cluster_to_class[cluster] for cluster in unlabeled_predictions]
)

# Step 7: Evaluate Performance on Unlabeled Data
# Compute accuracy for the unlabeled subset
unlabeled_acc = accuracy_score(y_unlabeled, final_unlabeled_predictions)
print(f"Accuracy on unlabeled data: {unlabeled_acc * 100:.2f}%")
print(f"Performance Improvement (Labeled vs Unlabeled): {labeled_acc - unlabeled_acc:.2f}")

# Step 8: Visualize the Clusters
# Combine predictions
combined_predictions = np.zeros_like(kmeans_labels)
combined_predictions[:len(X_labeled)] = labeled_predictions
combined_predictions[len(X_labeled):] = final_unlabeled_predictions

# Add the cluster labels to the dataframe for visualization
df['Cluster'] = combined_predictions

# Plot the clusters
plt.figure(figsize=(10, 7))
sns.scatterplot(
    x=df['mean radius'],
    y=df['mean smoothness'],
    hue=df['Cluster'],
    palette="viridis",
    s=100,
    alpha=0.8,
)
plt.title("Clusters Predicted by GMM")
plt.xlabel("Mean Radius")
plt.ylabel("Mean Smoothness")
plt.legend(title="Cluster")
plt.grid()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.cluster import KMeans

# Load and preprocess the dataset
cancer = load_breast_cancer()
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)

# Selecting relevant features
df = df[['mean radius', 'mean smoothness']]

# Scale the features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df)

# Generate Pseudo-Labels using K-Means
kmeans = KMeans(n_clusters=2, random_state=42)
kmeans_labels = kmeans.fit_predict(scaled_features)

# Different labeled to unlabeled ratios
ratios = [0.5, 0.2, 0.8] # 50:50, 20:80, 80:20
performance_results = []

for ratio in ratios:
    print(f"\n--- Evaluating GMM with {int(ratio * 100)}% labeled data ---")

    # Split dataset
    X_labeled, X_unlabeled, y_labeled, y_unlabeled = train_test_split(
        scaled_features, kmeans_labels, test_size=1-ratio, random_state=42, stratify=kmeans_labels
    )

    # Train Gaussian Mixture Model (GMM)
    gmm = GaussianMixture(n_components=2, random_state=42)
    gmm.fit(X_labeled)

    # Evaluate on labeled data
    labeled_predictions = gmm.predict(X_labeled)
    labeled_acc = accuracy_score(y_labeled, labeled_predictions)
    print(f"Accuracy on labeled data: {labeled_acc * 100:.2f}%")

    # Predict clusters for unlabeled data
    unlabeled_predictions = gmm.predict(X_unlabeled)

    # Map clusters to pseudo-labels
    cluster_to_class = {}
    for cluster in np.unique(unlabeled_predictions):
        mask = unlabeled_predictions == cluster
        majority_class = np.bincount(y_unlabeled[mask]).argmax()
        cluster_to_class[cluster] = majority_class

    final_unlabeled_predictions = np.array(
        [cluster_to_class[cluster] for cluster in unlabeled_predictions]
    )

    # Evaluate performance on unlabeled data
    unlabeled_acc = accuracy_score(y_unlabeled, final_unlabeled_predictions)
    print(f"Accuracy on unlabeled data: {unlabeled_acc * 100:.2f}%")
    print(f"Performance Difference (Labeled vs Unlabeled): {labeled_acc - unlabeled_acc:.2f}")

    performance_results.append((ratio, labeled_acc, unlabeled_acc))

# Visualizing the performance impact
ratios, labeled_accs, unlabeled_accs = zip(*performance_results)

plt.figure(figsize=(8, 5))
plt.plot([r * 100 for r in ratios], labeled_accs, marker='o', label="Labeled Data Accuracy")
plt.plot([r * 100 for r in ratios], unlabeled_accs, marker='s', label="Unlabeled Data Accuracy")
plt.xlabel("Labeled Data Percentage (%)")
plt.ylabel("Accuracy")
plt.title("Effect of Labeled Data Ratio on GMM Performance")
plt.legend()
plt.grid()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.cluster import KMeans

# Load and preprocess the dataset
cancer = load_breast_cancer()
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)

# Selecting relevant features
df = df[['mean radius', 'mean smoothness']]

# Scale the features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df)

# Experimenting with different numbers of clusters
cluster_counts = # Number of clusters to evaluate
performance_results = []

for n_clusters in cluster_counts:
    print(f"\n--- Evaluating GMM with {n_clusters} Clusters ---")

    # Generate Pseudo-Labels using K-Means
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    kmeans_labels = kmeans.fit_predict(scaled_features)

    # Split dataset into labeled and unlabeled subsets (fixed at 70% unlabeled, 30% labeled)
    X_labeled, X_unlabeled, y_labeled, y_unlabeled = train_test_split(
        scaled_features, kmeans_labels, test_size=0.7, random_state=42, stratify=kmeans_labels
    )

    # Train Gaussian Mixture Model (GMM)
    gmm = GaussianMixture(n_components=n_clusters, random_state=42)
    gmm.fit(X_labeled)

    # Evaluate on labeled data
    labeled_predictions = gmm.predict(X_labeled)
    labeled_acc = accuracy_score(y_labeled, labeled_predictions)
    print(f"Accuracy on labeled data: {labeled_acc * 100:.2f}%")

    # Predict clusters for unlabeled data
    unlabeled_predictions = gmm.predict(X_unlabeled)

    # Map clusters to pseudo-labels
    cluster_to_class = {}
    for cluster in np.unique(unlabeled_predictions):
        mask = unlabeled_predictions == cluster
        majority_class = np.bincount(y_unlabeled[mask]).argmax()
        cluster_to_class[cluster] = majority_class

    final_unlabeled_predictions = np.array(
        [cluster_to_class[cluster] for cluster in unlabeled_predictions]
    )

    # Evaluate performance on unlabeled data
    unlabeled_acc = accuracy_score(y_unlabeled, final_unlabeled_predictions)
    print(f"Accuracy on unlabeled data: {unlabeled_acc * 100:.2f}%")
    print(f"Performance Difference (Labeled vs Unlabeled): {labeled_acc - unlabeled_acc:.2f}")

    performance_results.append((n_clusters, labeled_acc, unlabeled_acc))

# Visualizing the impact of increasing clusters
clusters, labeled_accs, unlabeled_accs = zip(*performance_results)

plt.figure(figsize=(8, 5))
plt.plot(clusters, labeled_accs, marker='o', label="Labeled Data Accuracy")
plt.plot(clusters, unlabeled_accs, marker='s', label="Unlabeled Data Accuracy")
plt.xlabel("Number of Clusters")
plt.ylabel("Accuracy")
plt.title("Effect of Increasing Clusters on GMM Performance")
plt.legend()
plt.grid()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.cluster import KMeans

# Load and preprocess the dataset
cancer = load_breast_cancer()
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)

# Selecting relevant features
df = df[['mean radius', 'mean smoothness']]

# Scale the features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df)

# Generate Pseudo-Labels using K-Means
kmeans = KMeans(n_clusters=2, random_state=42)
kmeans_labels = kmeans.fit_predict(scaled_features)

# Split dataset into labeled and unlabeled subsets (30% labeled, 70% unlabeled)
X_labeled, X_unlabeled, y_labeled, y_unlabeled = train_test_split(
    scaled_features, kmeans_labels, test_size=0.7, random_state=42, stratify=kmeans_labels
)

# Experimenting with different covariance types
covariance_types = ['full', 'tied', 'diag', 'spherical']
performance_results = []

for cov_type in covariance_types:
    print(f"\n--- Evaluating GMM with Covariance Type: {cov_type} ---")

    # Train Gaussian Mixture Model (GMM)
    gmm = GaussianMixture(n_components=2, covariance_type=cov_type, random_state=42)
    gmm.fit(X_labeled)

    # Evaluate on labeled data
    labeled_predictions = gmm.predict(X_labeled)
    labeled_acc = accuracy_score(y_labeled, labeled_predictions)
    print(f"Accuracy on labeled data: {labeled_acc * 100:.2f}%")

    # Predict clusters for unlabeled data
    unlabeled_predictions = gmm.predict(X_unlabeled)

    # Map clusters to pseudo-labels
    cluster_to_class = {}
    for cluster in np.unique(unlabeled_predictions):
        mask = unlabeled_predictions == cluster
        majority_class = np.bincount(y_unlabeled[mask]).argmax()
        cluster_to_class[cluster] = majority_class

    final_unlabeled_predictions = np.array(
        [cluster_to_class[cluster] for cluster in unlabeled_predictions]
    )

    # Evaluate performance on unlabeled data
    unlabeled_acc = accuracy_score(y_unlabeled, final_unlabeled_predictions)
    print(f"Accuracy on unlabeled data: {unlabeled_acc * 100:.2f}%")
    print(f"Performance Difference (Labeled vs Unlabeled): {labeled_acc - unlabeled_acc:.2f}")

    performance_results.append((cov_type, labeled_acc, unlabeled_acc))

# Visualizing the impact of covariance types
cov_types, labeled_accs, unlabeled_accs = zip(*performance_results)

x = np.arange(len(cov_types))
width = 0.35

plt.figure(figsize=(10, 6))
plt.bar(x - width/2, labeled_accs, width, label='Labeled Data Accuracy', color='skyblue')
plt.bar(x + width/2, unlabeled_accs, width, label='Unlabeled Data Accuracy', color='salmon')

plt.xlabel("Covariance Type")
plt.ylabel("Accuracy")
plt.title("Effect of Covariance Type on GMM Performance")
plt.xticks(x, cov_types)
plt.ylim(0, 1.1)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()